In [87]:
%%capture
!pip install unidecode

In [88]:
import torch
from google.colab import drive

drive.mount("/drive")

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [89]:
from unidecode import unidecode

In [90]:
!ls  /drive/MyDrive/names

Arabic.txt   English.txt  Irish.txt	Polish.txt	Spanish.txt
Chinese.txt  French.txt   Italian.txt	Portuguese.txt	Vietnamese.txt
Czech.txt    German.txt   Japanese.txt	Russian.txt
Dutch.txt    Greek.txt	  Korean.txt	Scottish.txt


In [91]:
!cat /drive/MyDrive/names/Arabic.txt

Khoury
Nahas
Daher
Gerges
Nazari
Maalouf
Gerges
Naifeh
Guirguis
Baba
Sabbagh
Attia
Tahan
Haddad
Aswad
Najjar
Dagher
Maloof
Isa
Asghar
Nader
Gaber
Abboud
Maalouf
Zogby
Srour
Bahar
Mustafa
Hanania
Daher
Tuma
Nahas
Saliba
Shamoon
Handal
Baba
Amari
Bahar
Atiyeh
Said
Khouri
Tahan
Baba
Mustafa
Guirguis
Sleiman
Seif
Dagher
Bahar
Gaber
Harb
Seif
Asker
Nader
Antar
Awad
Srour
Shadid
Hajjar
Hanania
Kalb
Shadid
Bazzi
Mustafa
Masih
Ghanem
Haddad
Isa
Antoun
Sarraf
Sleiman
Dagher
Najjar
Malouf
Nahas
Naser
Saliba
Shamon
Malouf
Kalb
Daher
Maalouf
Wasem
Kanaan
Naifeh
Boutros
Moghadam
Masih
Sleiman
Aswad
Cham
Assaf
Quraishi
Shalhoub
Sabbag
Mifsud
Gaber
Shammas
Tannous
Sleiman
Bazzi
Quraishi
Rahal
Cham
Ghanem
Ghanem
Naser
Baba
Shamon
Almasi
Basara
Quraishi
Bata
Wasem
Shamoun
Deeb
Touma
Asfour
Deeb
Hadad
Naifeh
Touma
Bazzi
Shamoun
Nahas
Haddad
Arian
Kouri
Deeb
Toma
Halabi
Nazari
Saliba
Fakhoury
Hadad
Baba
Mansour
Sayegh
Antar
Deeb
Morcos
Shalhoub
Sarraf
Amari
Wasem
Ganim
Tuma
Fakhoury
Hadad
Hakimi
Nader
Sa

In [92]:
import os
from glob import glob

In [93]:
root_dir = "/drive/MyDrive/names"
file_names = glob("*.txt", root_dir=root_dir)
unique_labels = sorted([os.path.splitext(file_name)[0] for file_name in file_names])
n_labels = len(unique_labels)

idx2label = {idx:label for idx, label in enumerate(unique_labels)}
label2idx = {label:idx  for idx, label in idx2label.items()}


In [94]:
idx2label

{0: 'Arabic',
 1: 'Chinese',
 2: 'Czech',
 3: 'Dutch',
 4: 'English',
 5: 'French',
 6: 'German',
 7: 'Greek',
 8: 'Irish',
 9: 'Italian',
 10: 'Japanese',
 11: 'Korean',
 12: 'Polish',
 13: 'Portuguese',
 14: 'Russian',
 15: 'Scottish',
 16: 'Spanish',
 17: 'Vietnamese'}

In [95]:
def replace(name, chars, target):
  for char in chars:
    name = name.replace(char, target)
  return name

In [96]:
X_names = []
Y_labels = []

for file_name in file_names:
  with open(os.path.join(root_dir, file_name), "rt", encoding='utf-8') as f:
    for line in f:
      name = line.strip().lower()
      name = unidecode(name)

      if name == 'to the first page':
        continue

      name = replace(name, [",", "1", "/b", ":", "\xa0"], '')
      name = replace(name, [''], '')

      X_names.append(name)
      Y_labels.append(os.path.splitext(file_name)[0])

In [97]:
pad_token = '.'
pad_token_id = 0

unique_chars = [pad_token] + sorted(set(''.join(X_names)))
idx2char = {idx:char for idx, char in enumerate(unique_chars)}
char2idx = {char:idx for idx, char in idx2char.items()}

def encode(name: str) -> list[int]:
  return [char2idx[char] for char in name]

def decode(ids: list[int]) -> str:
  return ''.join(idx2char[i] for i in ids)


In [98]:
idx2char

{0: '.',
 1: ' ',
 2: "'",
 3: '-',
 4: 'a',
 5: 'b',
 6: 'c',
 7: 'd',
 8: 'e',
 9: 'f',
 10: 'g',
 11: 'h',
 12: 'i',
 13: 'j',
 14: 'k',
 15: 'l',
 16: 'm',
 17: 'n',
 18: 'o',
 19: 'p',
 20: 'q',
 21: 'r',
 22: 's',
 23: 't',
 24: 'u',
 25: 'v',
 26: 'w',
 27: 'x',
 28: 'y',
 29: 'z'}

In [99]:
Y = [label2idx[(label)] for label in Y_labels]
X = [encode(name) for name in X_names]

In [100]:
for x, x_name, y, y_label in zip(X[:5], X_names[:5], Y[:5], Y_labels[:5]):
  print(f"{str(x):<50} -> {x_name:30} \t\t{y} -> {y_label}")

[4, 5, 4, 5, 14, 18]                               -> ababko                         		14 -> Russian
[4, 5, 4, 8, 25]                                   -> abaev                          		14 -> Russian
[4, 5, 4, 10, 28, 4, 17]                           -> abagyan                        		14 -> Russian
[4, 5, 4, 12, 7, 24, 15, 12, 17]                   -> abaidulin                      		14 -> Russian
[4, 5, 4, 12, 7, 24, 15, 15, 12, 17]               -> abaidullin                     		14 -> Russian


In [101]:
from sklearn.model_selection import train_test_split

In [102]:
X_tr, X_ts, Y_tr, Y_ts = train_test_split(X, Y, test_size=0.2, stratify=Y)

In [103]:
from torch.utils.data import Dataset, DataLoader

class NamesDataset(Dataset):
  def __init__(self, X, Y):
    self.X = X
    self.Y = Y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.Y[idx]


Dtr = NamesDataset(X_tr, Y_tr)
Dts = NamesDataset(X_ts, Y_ts)



In [104]:
len(Dtr)

16040

In [105]:
len(Dts)

4011

In [106]:
Dtr[0]

([25, 4, 15, 14, 18, 25], 14)

In [107]:
global_max_n = 20

def collate_fn(batch):
  X_batch, Y_batch = zip(*batch)
  max_len = max(len(x) for x in X_batch)
  max_len = global_max_n if max_len > global_max_n else max_len

  padded_X = [x + [pad_token_id] * (max_len - len(x)) for x in X_batch]

  return torch.tensor(padded_X), torch.tensor(Y_batch)

Dltr = DataLoader(Dtr, batch_size=4, shuffle=True, drop_last=True, collate_fn=collate_fn)
Dlts = DataLoader(Dts, batch_size=4, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [108]:
for batch in Dltr:
  print(batch)
  break

(tensor([[ 5,  4, 29,  0,  0,  0,  0,  0,  0],
        [10, 18, 21, 22, 23,  0,  0,  0,  0],
        [19, 15, 12, 22,  8, 14,  0,  0,  0],
        [ 5, 15,  4, 17,  6, 11,  8, 23, 23]]), tensor([ 0, 14,  2,  5]))


In [137]:
# Define the model
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class NamesClassifier(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.emb = nn.Embedding(self.config.vocab_size, self.config.n_embd)
    self.conv = nn.Conv1d(self.config.n_embd, self.config.n_conv_channels, self.config.kernel_size)
    self.max_pool = nn.AdaptiveMaxPool1d(1)
    self.drop = nn.Dropout(self.config.drop_rate)
    self.fc = nn.Linear(self.config.n_conv_channels, self.config.n_labels)

  def forward(self, x):
    x = self.emb(x)
    x = x.permute(0, 2, 1)
    x = self.conv(x)
    x = self.max_pool(x)
    x = self.drop(x.squeeze())
    x = self.fc(x)
    return x


@dataclass
class Config:
  vocab_size: int
  n_embd: int
  n_conv_channels: int
  kernel_size: int
  drop_rate: float
  n_labels: int


config = Config(vocab_size=29, n_embd=16, n_conv_channels=32, kernel_size=3, drop_rate=0.5, n_labels=18)
model = NamesClassifier(config)

In [140]:
x.shape

torch.Size([4, 12])

In [139]:
model(x).shape

torch.Size([4, 18])

In [111]:
n_epochs = 3

for epoch in range(1, n_epochs+1):
  for x, y in Dltr:
    break
  break
